# Run the recommendation pipeline

This job executes the notebooks in dependency order:
1. `00_get_data.ipynb` collects Twitch data and exports snapshots
2. `01_set_baseline.ipynb` creates the weighted baseline ranking
3. `02_compare_models.ipynb` runs staged filtering and the model pipeline
4. `03_metadata_embeddings.ipynb` trains metadata embeddings and cosine-similarity recommendations

Set Twitch and Postgres environment variables before running. The job skips interactive Twitch OAuth; provide `TWITCH_USER_ACCESS_TOKEN` for user-specific follows and VODs.

In [9]:
import importlib
import os
from pathlib import Path

nbformat = importlib.import_module("nbformat")
NotebookClient = importlib.import_module("nbclient.client").NotebookClient

notebooks_dir = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
pipeline = [
    notebooks_dir / "00_get_data.ipynb",
    notebooks_dir / "01_set_baseline.ipynb",
    notebooks_dir / "02_compare_models.ipynb",
    notebooks_dir / "03_metadata_embeddings.ipynb",
]
missing = [str(path) for path in pipeline if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing pipeline notebooks: {missing}")

previous_job_mode = os.environ.get("TWITCH_PIPELINE_JOB")
os.environ["TWITCH_PIPELINE_JOB"] = "1"
try:
    for notebook_path in pipeline:
        print(f"Running {notebook_path.name}...")
        with notebook_path.open("r", encoding="utf-8") as handle:
            notebook = nbformat.read(handle, as_version=4)
        client = NotebookClient(
            notebook,
            timeout=None,
            kernel_name="python3",
            resources={"metadata": {"path": str(notebooks_dir)}},
        )
        client.execute()
        with notebook_path.open("w", encoding="utf-8") as handle:
            nbformat.write(notebook, handle)
        print(f"Completed {notebook_path.name}")
finally:
    if previous_job_mode is None:
        os.environ.pop("TWITCH_PIPELINE_JOB", None)
    else:
        os.environ["TWITCH_PIPELINE_JOB"] = previous_job_mode

print("Recommendation pipeline completed successfully.")

Running 00_get_data.ipynb...
Completed 00_get_data.ipynb
Running 01_set_baseline.ipynb...
Completed 01_set_baseline.ipynb
Running 02_compare_models.ipynb...
Completed 02_compare_models.ipynb
Running 03_metadata_embeddings.ipynb...
Completed 03_metadata_embeddings.ipynb
Recommendation pipeline completed successfully.


In [10]:
import importlib
import json
import os
from datetime import datetime, timezone

try:
    load_dotenv = importlib.import_module("dotenv").load_dotenv
    repo_root = notebooks_dir.parent if notebooks_dir.name == "notebooks" else notebooks_dir
    load_dotenv(repo_root / ".env.local")
    load_dotenv(repo_root / ".env")
except ImportError:
    pass

DATABASE_URL = os.environ.get("DATABASE_URL")
if DATABASE_URL:
    psycopg = importlib.import_module("psycopg")
    Jsonb = importlib.import_module("psycopg.types.json").Jsonb
    embedding_path = notebooks_dir / "data" / "embedding_top_50.jsonl"
    if not embedding_path.exists():
        raise FileNotFoundError(f"Embedding recommendations not found: {embedding_path}")

    collection_time = datetime.now(timezone.utc)
    create_table_sql = """
    CREATE TABLE IF NOT EXISTS twitch_recommendation_records (
        dataset TEXT NOT NULL,
        record_key TEXT NOT NULL,
        collected_at TIMESTAMPTZ NOT NULL,
        payload JSONB NOT NULL,
        PRIMARY KEY (dataset, record_key, collected_at)
    )
    """
    insert_sql = """
    INSERT INTO twitch_recommendation_records (dataset, record_key, collected_at, payload)
    VALUES (%s, %s, %s, %s)
    ON CONFLICT (dataset, record_key, collected_at)
    DO UPDATE SET payload = EXCLUDED.payload
    """
    with embedding_path.open("r", encoding="utf-8") as handle:
        recommendation_records = [json.loads(line) for line in handle if line.strip()]
    rows_to_insert = [
        ("embedding_top_50", str(record.get("id") or index), collection_time, Jsonb(record))
        for index, record in enumerate(recommendation_records)
    ]
    with psycopg.connect(DATABASE_URL, prepare_threshold=None) as connection:
        with connection.cursor() as cursor:
            cursor.execute(create_table_sql)
            cursor.executemany(insert_sql, rows_to_insert)
    print(f"Exported {len(rows_to_insert)} embedding recommendations to Postgres")
else:
    print("DATABASE_URL is not configured; embedding recommendations remain in JSONL only.")

Exported 50 embedding recommendations to Postgres


In [11]:
from pathlib import Path

import pandas as pd
from IPython.display import display

notebooks_dir = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
model_outputs = {
    "Baseline model": ("baseline_top_50.jsonl", "y_score"),
    "Complex model": ("complex_model_top_50.jsonl", "complex_score"),
    "Metadata embedding model": ("embedding_top_50.jsonl", "embedding_score"),
}

for model_name, (filename, score_column) in model_outputs.items():
    output_path = notebooks_dir / "data" / filename
    if not output_path.exists():
        print(f"{model_name}: output not found at {output_path}. Run the pipeline cell first.")
        continue
    results_df = pd.read_json(output_path, lines=True)
    columns = [
        column for column in [
            "rank", "user_name", "game_name", "title", "viewer_count",
            "y_score", "complex_score", "metadata_similarity", "embedding_score",
        ] if column in results_df.columns
    ]
    print(f"\n{model_name} ({len(results_df)} recommendations)")
    display(results_df[columns])


Baseline model (50 recommendations)


,rank,user_name,game_name,title,viewer_count,y_score
0,1,Kamet0,League of Legends,KARMINE CORP 0 vs 0 MOVISTAR KOI | WORLDS 2026...,144733,0.614814
1,2,Caedrel,League of Legends,🔴WORLDS 2026 KOI VS KC ELIMINATION🔴,209248,0.592816
2,3,ibai,League of Legends,#WORLDS2026 | MKOI vs KC | EL PARTIDO DEL AÑO ...,93931,0.564505
3,4,otplol_,League of Legends,WORLDS - KC vs MKOI - POUR UNE PLACE EN QUART ...,48617,0.494661
4,5,Riot Games,League of Legends,KC vs MKOI | WORLDS 2026 | Swiss Stage Day 8,42921,0.482247
5,6,Trymacs,Minecraft,100.000 TNT Exploion Craft Attack Rekord aufst...,31078,0.482086
6,7,EDISON,Just Chatting,🎃КОНКУРС КОСТЮМОВ НА ХЕЛЛОУИН! [сейчас],29750,0.462975
7,8,ow_esports,Overwatch,[DROPS] OWCS 2026 | Stage 3 Playoffs Day 2,21991,0.461264
8,9,morphe_ya,IRL,Забег день 4. ХЭЛЛОУИНСКИ МОРДЫ @dangerlyoha @...,23381,0.458722
9,10,Warframe,Warframe,"PMCF Quest Finale | Saturday, October 31 @ 12P...",27728,0.457499



Complex model (50 recommendations)


,rank,user_name,game_name,title,viewer_count,complex_score
0,1,Kamet0,League of Legends,KARMINE CORP 0 vs 0 MOVISTAR KOI | WORLDS 2026...,144733,0.563770
1,2,ibai,League of Legends,#WORLDS2026 | MKOI vs KC | EL PARTIDO DEL AÑO ...,93931,0.562419
2,3,otplol_,League of Legends,WORLDS - KC vs MKOI - POUR UNE PLACE EN QUART ...,48617,0.562419
3,4,Riot Games,League of Legends,KC vs MKOI | WORLDS 2026 | Swiss Stage Day 8,42921,0.555432
4,5,EDISON,Just Chatting,🎃КОНКУРС КОСТЮМОВ НА ХЕЛЛОУИН! [сейчас],29750,0.514718
5,6,morphe_ya,IRL,Забег день 4. ХЭЛЛОУИНСКИ МОРДЫ @dangerlyoha @...,23381,0.514087
6,7,Caedrel,League of Legends,🔴WORLDS 2026 KOI VS KC ELIMINATION🔴,209248,0.510007
7,8,Trymacs,Minecraft,100.000 TNT Exploion Craft Attack Rekord aufst...,31078,0.505131
8,9,Warframe,Warframe,"PMCF Quest Finale | Saturday, October 31 @ 12P...",27728,0.504666
9,10,ow_esports,Overwatch,[DROPS] OWCS 2026 | Stage 3 Playoffs Day 2,21991,0.502280



Metadata embedding model (50 recommendations)


,rank,user_name,game_name,title,viewer_count,metadata_similarity,embedding_score
0,1,Caedrel,League of Legends,🔴WORLDS 2026 KOI VS KC ELIMINATION🔴,209248,0,0.30000
1,2,Kamet0,League of Legends,KARMINE CORP 0 vs 0 MOVISTAR KOI | WORLDS 2026...,144733,0,0.29994
2,3,ibai,League of Legends,#WORLDS2026 | MKOI vs KC | EL PARTIDO DEL AÑO ...,93931,0,0.29988
3,4,otplol_,League of Legends,WORLDS - KC vs MKOI - POUR UNE PLACE EN QUART ...,48617,0,0.29982
4,5,Riot Games,League of Legends,KC vs MKOI | WORLDS 2026 | Swiss Stage Day 8,42921,0,0.29976
5,6,Trymacs,Minecraft,100.000 TNT Exploion Craft Attack Rekord aufst...,31078,0,0.29970
6,7,EDISON,Just Chatting,🎃КОНКУРС КОСТЮМОВ НА ХЕЛЛОУИН! [сейчас],29750,0,0.29964
7,8,Warframe,Warframe,"PMCF Quest Finale | Saturday, October 31 @ 12P...",27728,0,0.29958
8,9,morphe_ya,IRL,Забег день 4. ХЭЛЛОУИНСКИ МОРДЫ @dangerlyoha @...,23381,0,0.29952
9,10,ow_esports,Overwatch,[DROPS] OWCS 2026 | Stage 3 Playoffs Day 2,21991,0,0.29946


## Main outputs

The job writes refreshed JSONL snapshots under `notebooks/data/`, including `baseline_top_50.jsonl`, `complex_model_top_50.jsonl`, and `embedding_top_50.jsonl`. Notebook 00 also exports the collected records to Postgres when `DATABASE_URL` is configured.